In [1]:
import gymnasium
import numpy as np
import torch
import yaml
from PIL import Image
from torchvision.transforms import v2
from einops import rearrange

from metaworld_env import setup_metaworld_env

from model.encoder import EncoderNet
from model.actor import DDPGActorNet
from model.critic import CriticNet

In [2]:
task_name = "windowopen"
seed = 100

In [3]:
def setup_env(task_name:str, seed:int, render_mode:str|None = None) -> gymnasium.Env:
    env = setup_metaworld_env(task_name, seed, render_mode)
    env = gymnasium.wrappers.RecordEpisodeStatistics(env)
    #env = gymnasium.wrappers.AddRenderObservation(env, render_only=True)
    #env = gymnasium.wrappers.FrameStackObservation(env, 2)

    return env

In [4]:
with open("configs/ddpg.yaml", "r") as file:
    config = yaml.safe_load(file)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
envs = setup_env(task_name, 5000, "rgb_array")

obs_dim = envs.observation_space.shape
action_dim = envs.action_space.shape

In [8]:
encoder = EncoderNet(np.prod(obs_dim), config["num_blocks"], config["encoder_layers"]).to(device)
actor = DDPGActorNet(encoder.dim, np.prod(action_dim), config["actor_layers"]).to(device)
critic = CriticNet(encoder.dim, np.prod(action_dim), config["critic_layers"]).to(device)

encoder_weight, actor_weight, critic_weight = torch.load(f"weights/ddpg/{task_name}/actor_{seed}_100.pt", weights_only=True)
encoder.load_state_dict(encoder_weight)
actor.load_state_dict(actor_weight)
critic.load_state_dict(critic_weight)

encoder = encoder.eval()
actor = actor.eval()

transform = v2.Compose([
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize((0.485, 0.456, 0.406, 0.485, 0.456, 0.406), (0.229, 0.224, 0.225, 0.229, 0.224, 0.225)),
        ])

In [9]:
def preprpcess(obs_batch):
    obs_batch = rearrange(obs_batch, "b l h w c -> b (l c) h w")
    obs_batch = transform(obs_batch)
    return obs_batch

@torch.no_grad()
def get_action(obs_batch, deterministic, random):

    #obs_batch = torch.as_tensor(obs_batch, dtype=torch.float32).to(device)
    obs_batch = torch.as_tensor(obs_batch).unsqueeze(0).to(device)
    #obs_batch = preprpcess(obs_batch)
    obs_batch = encoder(obs_batch)
    dist = actor(obs_batch, 1)
    if deterministic:
        action = dist.mean
    else:    
        action = dist.sample(clip=None)

        if random:
            action.uniform_(-1, 1)
    
    action = action.cpu().numpy()
    
    return action.tolist()

In [10]:
obs, info = envs.reset()
success = 0
imgs = []
for i in range(500):
    imgs.append(Image.fromarray(envs.render()))
    action = get_action(obs, True, False)
    next_obs, reward, done, timeout, info = envs.step(action[0])
    success += info["success"]
    obs = next_obs

imgs[0].save('output.gif',
               save_all=True,
               append_images=imgs[1:],  # Remaining frames
               duration=100,              # Duration per frame in ms
               loop=0)  

In [ ]:
success_buffer = []
for _ in range(100):
    success = 0
    obs, info = envs.reset()

    for i in range(500):
        action = get_action(obs, True, False)
        next_obs, reward, done, timeout, info = envs.step(action[0])
        success += info["success"]
        obs = next_obs
    
    success_buffer.append(success/500)

In [8]:
np.mean(success_buffer)

np.float64(0.70112)

In [17]:
obs_batch = torch.as_tensor(obs).unsqueeze(0).to(device)

feature = encoder(obs_batch)
feature_aug = torch.nn.functional.dropout(feature, p=0.5, training=True)

In [18]:
torch.nn.functional.cosine_similarity(feature, feature_aug)

tensor([0.6371], device='cuda:0', grad_fn=<SumBackward1>)